# Probability Concepts with Real-World Examples

## Real-World Applications of Probability

### Autocorrect Systems
- **How it works**:
  - When you type "recieve", it suggests "receive"
  - Uses word frequency statistics (unigram/bigram probabilities)


### Spam Filters
- **How it works**:
  - Flags messages like "Win 1M!" as spam
  - Uses Bayesian probability to update beliefs


## Probability Terminology Explained

### Key Concepts with Email Examples

#### Random Experiment
- **Definition**: A process that produces an outcome
- **Email example**: Classifying an email as spam/ham
- **Other examples**: Rolling a die, flipping a coin

#### Trial
- **Definition**: One execution of a random experiment
- **Email example**: Processing one email message
- **Other examples**: One coin flip, one die roll

#### Outcome
- **Definition**: The result of a trial
- **Email example**: "spam" or "not spam" classification
- **Other examples**: "heads", "tails", die face number

#### Sample Space
- **Definition**: All possible outcomes
- **Email example**: All possible email texts and classifications
- **Notation**: S = {"spam", "not spam"}


### Corpus in Natural Language Processing

#### Definition
A **corpus** (plural: corpora) is a large, structured collection of texts used for linguistic analysis and training language models.

# Exercise 1: Next-Word Prediction



## Build Your Own Predictor

### Challenge
Design an algorithm that predicts the next word given typing history.

**Example Input/Output:**
- Input: "I want to"
- Output:
  - "go" (45% probability)
  - "eat" (30% probability)
  - "sleep" (25% probability)

## Next-Word Prediction Pipeline

### Step-by-Step Process
1. **Analyze current phrase**  
   Example: "How are"

2. **Generate likely candidates**  
   Potential next words: "you", "doing", "we"

3. **Rank by conditional probability**  
   Calculate P(word | context) for each candidate


# Conceptual implementation
context = "How are"
candidates = ["you", "doing", "we"]

# Calculate probabilities

Probabilities

    "you": 0.72,    # P("you" | "How are")

    "doing": 0.18,   # P("doing" | "How are")

    "we": 0.10       # P("we" | "How are")


# N-gram Language Models

## Markov Assumption
The probability of a word depends only on the previous **k** words:

P(w_n | w_1...w_{n-1}) ≈ P(w_n | w_{n-k}...w_{n-1})


## Unigram Model
**Assumption**: Words are completely independent  
**Probability**:
P(w) = count(w) / total_words

**Example**: "I love you"

P("I love you") = P("I") × P("love") × P("you")


## Bigram Model
**Assumption**: Word depends on 1 previous word  
**Probability**:
P(w_n | w_{n-1}) = count(w_{n-1} w_n) / count(w_{n-1})

**Example**: "I love you"
P("I love you") = P("I") × P("love"|"I") × P("you"|"love")

## Trigram Model
**Assumption**: Word depends on 2 previous words  
**Probability**:
P(w_n | w_{n-2}, w_{n-1}) = count(w_{n-2} w_{n-1} w_n) / count(w_{n-2} w_{n-1})

**Example**: "I love you"
P("I love you") = P("I") × P("love"|"I") × P("you"|"I love")

## Comparison Table

| Model    | Dependency          | Example Probability            |
|----------|---------------------|--------------------------------|
| Unigram  | None                | P("love")                     |
| Bigram   | 1 previous word     | P("you"/"love")              |
| Trigram  | 2 previous words    | P("you"/"I love")            |





### Dataset
Using Shakespeare's sonnets (public domain):
```python
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [9]:
import urllib.request

# Download dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

('input.txt', <http.client.HTTPMessage at 0x2eb44fc7610>)

#Data Loading & Preprocessing

In [10]:
def load_data(filepath):
    """Load text file and return as string"""
    with open(filepath, 'r', encoding='utf-8') as file:
        text = file.read()
    return text

def preprocess(text):
    """
    Preprocess text by:
    1. Converting to lowercase
    2. Removing special characters
    3. Splitting into tokens (words)
    """
    import re

    # Convert to lowercase
    text = text.lower()


    text = re.sub(r"[^a-z0-9'\s]", "", text)

    # Split into tokens (words)
    tokens = text.split()

    return tokens

# TEST
text = load_data("input.txt")
tokens = preprocess(text)
print("First 50 tokens:", tokens[:50])

First 50 tokens: ['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', 'all', 'resolved', 'resolved', 'first', 'citizen', 'first', 'you', 'know', 'caius', 'marcius', 'is', 'chief', 'enemy', 'to', 'the', 'people', 'all', 'we', "know't", 'we', "know't", 'first', 'citizen', 'let', 'us']


# Model Implementation

In [11]:
def build_ngram_model(tokens, n=2):
    """
    Build n-gram model from tokens
    Example: For n=2 (bigram), returns {"i": {"am": 5, "have": 3}}
    """
    model = {}

    # Loop through tokens with sliding window
    for window in zip(*[tokens[i:] for i in range(n)]):

        context = " ".join(window[:-1])   # previous word(s)
        next_word = window[-1]            # predicted word

        if context not in model:
            model[context] = {}

        if next_word not in model[context]:
            model[context][next_word] = 0

        model[context][next_word] += 1

    return model


def predict_next_word(model, context, k=3):
    """
    Predict next words given context
    Returns: list of (word, probability) tuples
    """

    if context not in model:
        return []

    next_words = model[context]

    total = sum(next_words.values())

    probs = []
    for word, count in next_words.items():
        probability = count / total
        probs.append((word, probability))

    # Sort by probability (highest first)
    probs.sort(key=lambda x: x[1], reverse=True)

    return probs[:k]

#Testing

In [12]:
bigram_model = build_ngram_model(tokens, n=2)
print("Words after 'the':", predict_next_word(bigram_model, "the"))

Words after 'the': [('king', 0.024685459468068164), ('duke', 0.0168816690555821), ('world', 0.015766841853798376)]


# Exercise 2: Spam/Ham Classification



## Spam vs Ham Examples

<div style="display: flex;">
<div style="flex: 50%; padding: 10px;">

### Spam Examples
- "Claim your free \$1M"
- "You won an iPhone!"
- "Limited time offer!"
- "Click here to claim your prize"

</div>
<div style="flex: 50%; padding: 10px;">

### Ham Examples
- "Meeting at 3 PM"
- "Project update attached"
- "Let's discuss the proposal"
- "Quarterly report is ready"

</div>
</div>


### 1. Prior Probability (P(c))
**What it represents:**  
The baseline probability of a class before examining the message content.

**Example Calculation:**  

#### In 100 emails:
spam_count = 30

ham_count = 70

P_spam = spam_count/100  # 0.30

P_ham = ham_count/100    # 0.70

### 2. Likelihood (P(w|c))

**Definition:**  
The probability of observing a specific word given a particular class (spam/ham). This measures how characteristic a word is for a certain class.


#### Spam class: 25 occurrences out of 1000 total words
P_free_spam = 25/1000  # 0.025

#### Ham class: 2 occurrences out of 3000 total words
P_free_ham = 2/3000    # ≈0.00067

### 3. Posterior Probability

**Definition:**
The posterior probability **P(c|msg)** represents:
- The probability that a given message belongs to class *c* (spam/ham)
- The key quantity we compute to make classification decisions

### Bayesian Classification

### Decision Rule
The classifier selects the class with the highest posterior probability:

Where:
- `P(c|msg)` = Posterior probability (what we want to calculate)
- `P(msg|c)` = Likelihood of the message given the class
- `P(c)` = Prior probability of the class

### Practical Example: "free prize" message

**Given Probabilities:**

### Word probabilities
P_free_spam = 0.025

P_free_ham = 0.00067

P_prize_spam = 0.015

P_prize_ham = 0.001



### Class priors
P_spam = 0.3

P_ham = 0.7

### Calculation Steps
#### Compute Spam Probability:
P_msg_spam = P_free_spam * P_prize_spam * P_spam
           = 0.025 × 0.015 × 0.3
           ≈ 0.0001125

####Compute Ham Probability:

P_msg_ham = P_free_ham * P_prize_ham * P_ham
          = 0.00067 × 0.001 × 0.7
          ≈ 0.000000469

####Comparison:
0.0001125 (Spam) > 0.000000469 (Ham)

##Laplace Smoothing
**Problem**:  
P("xyz"|Spam) = 0 if "xyz" never appeared in training spam

**Solution**:  
$$ P_{\text{smooth}}(w|c) = \frac{\text{count}(w,c) + 1}{\text{count}(c) + V} $$

*Example*:  
- V = 10,000 words  
- count("prize", Spam) = 0  
- count(Spam) = 1000  
$$ P_{\text{smooth}}("prize"|Spam) = \frac{0+1}{1000+10000} \approx 0.00009 $$



## Stop Words in Text Processing

**Definition:** Common words (e.g., "the", "a", "and") that appear frequently but carry minimal meaning.

**Why Remove Them?**
1. **Meaning Preservation:**  
   "The quick brown fox" → "quick brown fox" (keeps core meaning)
   
2. **Improved Analysis:**  
   - Without: P("the"|spam)=0.101 vs P("the"|ham)=0.098 (useless)  
   - With: P("free"|spam)=0.025 vs P("free"|ham)=0.00067 (discriminative)

3. **Efficiency:**  
   Reduces vocabulary size by 30-40% (1000 words → 600 words)

**Example: Spam Detection**
```python
from sklearn.feature_extraction.text import CountVectorizer

### With stop words
text = ["You have won a free prize"]
vectorizer = CountVectorizer()
print("With stop words:", vectorizer.fit_transform(text).toarray())
### Output: [[1 1 1 1 1]] (for "you", "have", "won", "free", "prize")

### Without stop words
vectorizer = CountVectorizer(stop_words='english')
print("Without stop words:", vectorizer.fit_transform(text).toarray())
### Output: [[1 1 1]] (for "won", "free", "prize")

## Dataset Loading

In [13]:
# Import required libraries
import requests, zipfile, io

# Step 1: Download the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
response = requests.get(url)

# Step 2: Extract the zip file
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
zip_file.extractall("sms_spam_dataset")  # Extract into this folder

print("Dataset downloaded and extracted!")

# Step 3: Load the SMS data
messages = []
labels = []

with open("sms_spam_dataset/SMSSpamCollection", "r", encoding="utf-8") as f:
    for line in f:
        label, text = line.strip().split("\t", 1)  # Split by tab
        labels.append(label)
        messages.append(text)

# Step 4: Check first few examples
print("First message:", messages[0])
print("First label:", labels[0])
print("Total messages loaded:", len(messages))

Dataset downloaded and extracted!
First message: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
First label: ham
Total messages loaded: 5574


In [14]:
import requests, zipfile, io, re

# Step 1: Download and extract the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
zip_file.extractall("sms_spam_dataset")
print("Dataset downloaded and extracted!")

# Step 2: Load data
def load_sms_data(filepath):
    messages = []
    labels = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                labels.append(parts[0])
                messages.append(parts[1])
    return messages, labels

messages, labels = load_sms_data("sms_spam_dataset/SMSSpamCollection")
print("First message:", messages[0])
print("First label:", labels[0])

# Step 3: Preprocess text
def preprocess_text(text, remove_stopwords=True):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if remove_stopwords:
        stop_words = {
            'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves',
            'you', 'your', 'yours', 'yourself', 'yourselves', 'he', 'him',
            'his', 'himself', 'she', 'her', 'hers', 'herself', 'it', 'its',
            'itself', 'they', 'them', 'their', 'theirs', 'themselves',
            'what', 'which', 'who', 'whom', 'this', 'that', 'these',
            'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
            'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing',
            'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as',
            'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about',
            'against', 'between', 'into', 'through', 'during', 'before',
            'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in',
            'out', 'on', 'off', 'over', 'under', 'again', 'further',
            'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how',
            'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other',
            'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same',
            'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just',
            'don', 'should', 'now'
        }
        words = text.split()
        text = ' '.join([w for w in words if w not in stop_words])
    return text

# Step 4: Test preprocessing
print("Processed message:", preprocess_text(messages[0]))
print("Processed without stopwords:", preprocess_text(messages[0], remove_stopwords=True))

Dataset downloaded and extracted!
First message: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
First label: ham
Processed message: go jurong point crazy available bugis n great world la e buffet cine got amore wat
Processed without stopwords: go jurong point crazy available bugis n great world la e buffet cine got amore wat


#Calculate Prior Probabilities

In [15]:
def calculate_priors(labels):
    # TODO: Compute P(spam) = (#spam messages)/(total messages)
    # Count total messages
    total_messages = len(labels)

    # Count number of spam messages
    num_spam = sum(1 for label in labels if label == 'spam')

    # TODO: Compute P(ham) similarly
    # Count number of ham messages
    num_ham = sum(1 for label in labels if label == 'ham')

    # Compute prior probabilities
    prior_prob = {
        'spam': num_spam / total_messages,
        'ham' : num_ham / total_messages
    }

    return prior_prob

# Example usage
labels = ['ham', 'spam', 'ham', 'ham', 'spam', 'ham']
priors = calculate_priors(labels)
print(priors)

{'spam': 0.3333333333333333, 'ham': 0.6666666666666666}


#Count Words per Class

In [16]:
def count_words(messages, labels):
    # TODO: Populate word_counts[class][word] = frequency
    # Hint: Use preprocess_text() from earlier
    # Don't forget to update vocab!

    word_counts = {'spam': {}, 'ham': {}}
    vocab = set()

    for message, label in zip(messages, labels):
        # Preprocess message (remove punctuation, lowercase, optional stopwords)
        clean_message = preprocess_text(message, remove_stopwords=True)

        # Split into words
        words = clean_message.split()

        # Update word counts for this class
        for word in words:
            vocab.add(word)  # Add to vocabulary
            if word in word_counts[label]:
                word_counts[label][word] += 1
            else:
                word_counts[label][word] = 1

    return word_counts, vocab

# Example usage
messages = ["Free entry in 2 a wkly comp", "Hey, how are you?", "Win cash now!"]
labels = ['spam', 'ham', 'spam']

word_counts, vocab = count_words(messages, labels)
print("Word counts:", word_counts)
print("Vocabulary:", vocab)

Word counts: {'spam': {'free': 1, 'entry': 1, '2': 1, 'wkly': 1, 'comp': 1, 'win': 1, 'cash': 1}, 'ham': {'hey': 1}}
Vocabulary: {'hey', 'cash', 'entry', 'free', '2', 'wkly', 'win', 'comp'}


#Compute Word Probabilities (with Smoothing)

In [17]:
def calculate_word_probs(word_counts, vocab, k=1):
    # TODO: For each word in vocab:
    # P(word|spam) = (count in spam + k)/(total spam words + k*V)
    # V = vocabulary size (len(vocab))

    word_probs = {'spam': {}, 'ham': {}}
    V = len(vocab)  # Vocabulary size

    # Total words per class
    total_spam_words = sum(word_counts['spam'].values())
    total_ham_words  = sum(word_counts['ham'].values())

    for word in vocab:
        # Calculate P(word|spam) with Laplace smoothing
        word_probs['spam'][word] = (word_counts['spam'].get(word, 0) + k) / (total_spam_words + k * V)
        # Calculate P(word|ham) with Laplace smoothing
        word_probs['ham'][word]  = (word_counts['ham'].get(word, 0) + k) / (total_ham_words + k * V)

    return word_probs

# Example usage
messages = ["Free entry in 2 a wkly comp", "Hey, how are you?", "Win cash now!"]
labels = ['spam', 'ham', 'spam']

# Step 1: Count words
word_counts, vocab = count_words(messages, labels)

# Step 2: Compute word probabilities
word_probs = calculate_word_probs(word_counts, vocab, k=1)
print("Word probabilities (spam):", word_probs['spam'])
print("Word probabilities (ham):", word_probs['ham'])

Word probabilities (spam): {'hey': 0.06666666666666667, 'cash': 0.13333333333333333, 'entry': 0.13333333333333333, 'free': 0.13333333333333333, '2': 0.13333333333333333, 'wkly': 0.13333333333333333, 'win': 0.13333333333333333, 'comp': 0.13333333333333333}
Word probabilities (ham): {'hey': 0.2222222222222222, 'cash': 0.1111111111111111, 'entry': 0.1111111111111111, 'free': 0.1111111111111111, '2': 0.1111111111111111, 'wkly': 0.1111111111111111, 'win': 0.1111111111111111, 'comp': 0.1111111111111111}


#Predict Spam Probability

In [18]:
import math

def predict(message, prior_prob, word_probs):
    # TODO:
    # 1. Preprocess message and split into words
    words = preprocess_text(message, remove_stopwords=True).split()

    # 2. Initialize log_prob_spam = log(P(spam))
    log_prob_spam = math.log(prior_prob['spam'])
    log_prob_ham  = math.log(prior_prob['ham'])

    # 3. For each word:
    for word in words:
        # - if word in vocab: add log(P(word|spam))
        if word in word_probs['spam']:
            log_prob_spam += math.log(word_probs['spam'][word])
        if word in word_probs['ham']:
            log_prob_ham += math.log(word_probs['ham'][word])

    # 4. Convert log probabilities back using exp()
    prob_spam = math.exp(log_prob_spam)
    prob_ham  = math.exp(log_prob_ham)

    # 5. Return P(spam|message) = spam_prob/(spam_prob + ham_prob)
    probability = prob_spam / (prob_spam + prob_ham)
    return probability

# Example usage
labels = ['ham', 'spam', 'ham', 'ham', 'spam', 'ham']
messages = ["Free entry in 2 a wkly comp", "Hey, how are you?", "Win cash now!"]

# Step 1: Calculate priors
priors = calculate_priors(labels)

# Step 2: Count words and build vocab
word_counts, vocab = count_words(messages, labels)

# Step 3: Calculate word probabilities
word_probs = calculate_word_probs(word_counts, vocab, k=1)

# Step 4: Predict
test_message = "Free cash win now"
spam_prob = predict(test_message, priors, word_probs)
print(f"Probability this message is spam: {spam_prob:.4f}")

Probability this message is spam: 0.2244


In [20]:
# Test calculate_priors()
labels = ['spam', 'ham']
print(calculate_priors(labels))  # Should return {'spam': 0.5, 'ham': 0.5}

# Test count_words()
messages = ["free prize", "meeting today"]
word_counts, vocab = count_words(messages, labels)
print(word_counts['spam']['free'])  # Should be 1
print(len(vocab))  # Should be 4 (free, prize, meeting, today)

# Test calculate_word_probs()
word_probs = calculate_word_probs(word_counts, vocab, k=1)
print(round(word_probs['spam']['free'], 2))  # Should be 0.33

{'spam': 0.5, 'ham': 0.5}
1
4
0.33
